# Simple RAG with LangChain using the same PDFKeep `metagpt.pdf` in the same folder as this notebook.

In [ ]:
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters pypdf faiss-cpu python-dotenv

In [ ]:
import osfrom dotenv import load_dotenvload_dotenv()OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")if not OPENAI_API_KEY:    raise ValueError("OPENAI_API_KEY not found.")

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddingsfrom langchain_community.document_loaders import PyPDFLoaderfrom langchain_text_splitters import RecursiveCharacterTextSplitterfrom langchain_community.vectorstores import FAISSfrom langchain_core.prompts import ChatPromptTemplatefrom langchain.chains import create_retrieval_chainfrom langchain.chains.combine_documents import create_stuff_documents_chain

In [ ]:
llm = ChatOpenAI(    model="gpt-5.4-nano",    api_key=OPENAI_API_KEY)embeddings = OpenAIEmbeddings(    model="text-embedding-3-small",    api_key=OPENAI_API_KEY)

In [ ]:
loader = PyPDFLoader("metagpt.pdf")   # same PDF namedocs = loader.load()print("Pages loaded:", len(docs))

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(    chunk_size=1000,    chunk_overlap=200)splits = text_splitter.split_documents(docs)print("Chunks:", len(splits))

In [ ]:
vectorstore = FAISS.from_documents(splits, embeddings)retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [ ]:
prompt = ChatPromptTemplate.from_template("""Answer the question using only the context below.If the answer is not in the context, say you do not know.<context>{context}</context>Question: {input}""")

In [ ]:
document_chain = create_stuff_documents_chain(llm, prompt)rag_chain = create_retrieval_chain(retriever, document_chain)

In [ ]:
response = rag_chain.invoke({"input": "What is the summary of this paper?"})print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "How do agents share information with other agents?"})print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "Tell me about the ablation study results."})print(response["answer"])

In [ ]:
for i, doc in enumerate(response["context"], 1):    print(f"\n--- Source {i} ---")    print(doc.page_content[:1000])